# The RAG Pipeline (Chunking -> Vector Store -> Generation)

Objective:  
You will learn why "how you read" data (Chunking) matters as much as "what you read," and you will build a Retrieval Augmented Generation (RAG) pipeline.  
  



# Section 0: Setup & Prerequisites

Install the necessary libraries. You will likely need langchain, langchain-community, chromadb, and an embedding provider.

In [28]:
# 1. Install/Update dependencies
!pip install -q -U \
  torch \
  transformers \
  sentence-transformers \
  accelerate \
  bitsandbytes \
  langchain \
  langchain-core \
  langchain-community \
  langchain-text-splitters \
  langchain-huggingface \
  chromadb \
  pysqlite3-binary \
  huggingface_hub

# 2. Fix Colab's SQLite version issue (Must happen before importing chromadb)
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# 3. Check if GPU is available
import torch
if not torch.cuda.is_available():
    print("WARNING: You are running on CPU. Go to Runtime -> Change runtime type -> T4 GPU")
else:
    print(f" GPU Detected: {torch.cuda.get_device_name(0)}")

 GPU Detected: Tesla T4


In [3]:
import os
from google.colab import userdata

# Vector Store & Embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEndpoint
from langchain_community.vectorstores import Chroma

# LLM & Generation
from langchain_huggingface import HuggingFaceEndpoint

print("Libraries imported successfully.")

Libraries imported successfully.


# Section 1: Chunking Experiment

LLMs have context windows. We must slice our data. But if you slice a sentence in half, the meaning might be lost. Let's prove this.

In [5]:
# Load a text file of your choice
import os
filename = "data/my_chapter.txt"

os.makedirs(os.path.dirname(filename), exist_ok = True)

sample_text = """Dr. Aris Thorne adjusted the dials on his suit's life-support interface. The red dust of Sector 4 swirled around his boots, obscuring the solar panels of the habitat module behind him. He had been stationed on Mars for over fourteen months, and the routine was always the same: collect soil samples, run atmospheric tests, and document the agonizingly slow progress of the terraforming algae.

But today was different.

Nestled in the shadow of a jagged basalt outcropping, something was glowing. It wasn't the harsh, artificial LED blue of the rover's headlights, nor was it the sterile white of the habitat's beacons. It was a soft, pulsating bioluminescent green. Aris dropped to his knees, his heart hammering against his ribs.

He carefully brushed away the iron-rich sand. There, clinging to the rock, was a delicate fern-like structure. Its leaves uncurled slowly, reacting to the heat radiating from his gloves. Life. Native, independent, glowing life. He activated his shoulder comms, his voice trembling. "Base, this is Thorne. You're going to want to see this."""

with open(filename, "w") as f:
  f.write(sample_text)

def load_data(path):
    with open(path, 'r', encoding = 'utf-8') as f:
      return f.read()

raw_text = load_data(filename)

print(f"Loaded {len(raw_text)} characters.")

Loaded 1071 characters.


Implement a splitter that strictly cuts text every x characters, regardless of sentence boundaries.

In [6]:
def naive_splitter(text, chunk_size=500):
    """
    Splits text strictly by character count.
    Returns: List[str]
    """
    return_list = []

    for i in range(0, len(text), chunk_size):
        return_list.append(text[i:i + chunk_size])
    return return_list

naive_chunks = naive_splitter(raw_text,50)
print(naive_chunks)

["Dr. Aris Thorne adjusted the dials on his suit's l", 'ife-support interface. The red dust of Sector 4 sw', 'irled around his boots, obscuring the solar panels', ' of the habitat module behind him. He had been sta', 'tioned on Mars for over fourteen months, and the r', 'outine was always the same: collect soil samples, ', 'run atmospheric tests, and document the agonizingl', 'y slow progress of the terraforming algae.\n\nBut to', 'day was different.\n\nNestled in the shadow of a jag', 'ged basalt outcropping, something was glowing. It ', "wasn't the harsh, artificial LED blue of the rover", "'s headlights, nor was it the sterile white of the", " habitat's beacons. It was a soft, pulsating biolu", 'minescent green. Aris dropped to his knees, his he', 'art hammering against his ribs.\n\nHe carefully brus', 'hed away the iron-rich sand. There, clinging to th', 'e rock, was a delicate fern-like structure. Its le', 'aves uncurled slowly, reacting to the heat radiati', 'ng from his gloves. 

Use a library (like LangChain) to split by "separators" (Paragraphs $\rightarrow$ Sentences $\rightarrow$ Words) to preserve meaning.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 450,
    chunk_overlap = 90,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Split the raw_text using the splitter
semantic_chunks = text_splitter.split_text(raw_text)

Find a specific example where the Naive splitter broke a sentence in half, rendering it meaningless, but the Semantic splitter kept it intact

In [8]:
def find_broken_context(naive_list, semantic_list):
    """
    Print a side-by-side comparison of a specific segment where
    Naive failed and Semantic succeeded.
    """
    print("Naive Splitter: ")
    for i in range(len(naive_list)):
        print(f"Chunk {i}: {repr(naive_list[i])}")

    print("\n")
    print("Semantic Splitter: ")
    for i in range(len(semantic_list)):
        print(f"Chunk {i}: {repr(semantic_list[i])}")

find_broken_context(naive_chunks, semantic_chunks)

Naive Splitter: 
Chunk 0: "Dr. Aris Thorne adjusted the dials on his suit's l"
Chunk 1: 'ife-support interface. The red dust of Sector 4 sw'
Chunk 2: 'irled around his boots, obscuring the solar panels'
Chunk 3: ' of the habitat module behind him. He had been sta'
Chunk 4: 'tioned on Mars for over fourteen months, and the r'
Chunk 5: 'outine was always the same: collect soil samples, '
Chunk 6: 'run atmospheric tests, and document the agonizingl'
Chunk 7: 'y slow progress of the terraforming algae.\n\nBut to'
Chunk 8: 'day was different.\n\nNestled in the shadow of a jag'
Chunk 9: 'ged basalt outcropping, something was glowing. It '
Chunk 10: "wasn't the harsh, artificial LED blue of the rover"
Chunk 11: "'s headlights, nor was it the sterile white of the"
Chunk 12: " habitat's beacons. It was a soft, pulsating biolu"
Chunk 13: 'minescent green. Aris dropped to his knees, his he'
Chunk 14: 'art hammering against his ribs.\n\nHe carefully brus'
Chunk 15: 'hed away the iron-rich sand. Th

In a text cell below, explain why the overlap parameter in the recursive splitter is essential for retrieval tasks.

To ensure no piece of information is lost between two chunks, no idea is isolated from the other chunks and context of the story flows smoothly throughtout the chunks, making it easier for the search engine to retrieve and pinpoint the correct chunk.

# Section 2: Vector Storage

We will convert our semantic_chunks into vector embeddings and store them.

Initialize Embeddings & DB:
- You may use OpenAI Embeddings (if you have a key) or HuggingFace (all-MiniLM-L6-v2) for a free, local alternative.
- Use ChromaDB or FAISS as your store.

In [9]:
# Initialize Embedding Model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a Vector Store for `semantic_chunks`
# Hint: Look for `.from_texts` or `.from_documents` in the LangChain/Chroma documentation.

# Clean old databse if present
if 'vector_db' in globals():
  vector_db.delete_collection()

vector_db = Chroma.from_texts(
    texts=semantic_chunks,
    embedding=embedding_model,
)

print(f"Number of chunks in vector db : {vector_db._collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of chunks in vector db : 3


# Section 3: The MVP (Retrieval Loop)

We have the brain (LLM) and the memory (Vector DB). Now we need to wire them together.

Create a function that takes a user query, converts it to a vector, and finds the top 3 most relevant chunks from your database.

In [10]:
def retrieve_context(query, k=3):
    """
    Args:
        query (str): The user's question
        k (int): Number of chunks to retrieve
    Returns:
        List[str]: The top k context chunks
    """
    # TODO: Use your vector_db to perform a similarity search
    matched_docs = vector_db.similarity_search(query, k = k)
    text_chunks = [doc.page_content for doc in matched_docs]
    return text_chunks
    pass

# Test it
test_query = "What is the first line"
context_results = retrieve_context(test_query)
print(f"Retrieved {len(context_results)} chunks.")

print("Top matched chunk : ")
print(context_results[0])

Retrieved 3 chunks.
Top matched chunk : 
He carefully brushed away the iron-rich sand. There, clinging to the rock, was a delicate fern-like structure. Its leaves uncurled slowly, reacting to the heat radiating from his gloves. Life. Native, independent, glowing life. He activated his shoulder comms, his voice trembling. "Base, this is Thorne. You're going to want to see this.


Construct the final prompt. You must inject the retrieved context into the system prompt so the LLM answers only based on that data.

In [19]:
# TODO: Initialize your LLM (OpenAI, Anthropic, or a local Llama via Ollama/HuggingFace)
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HF_TOKEN")

llm = HuggingFaceEndpoint(
    repo_id="google/flan-t5-large",
    task="text2text-generation", # This completely bypasses the 'conversational' server bug!
    max_new_tokens=256,
    temperature=0.1,
)

def generate_answer(query):
    # 1. Retrieve context
    context_chunks = retrieve_context(query)
    context_str = "\n".join(context_chunks)

    # 2. Construct the Prompt
    # Constraint: The prompt must instruct the LLM to say "I don't know" if the info isn't in the chunks.
    prompt = f""" You are a helpful assistant. Use the following pieces of retrieved context to answer user's question.
    If the answer is not explicitly contained in the context, exactly say "I don't know", without making up your answer.
    Context: {context_str}

    Question: {query}

    Answer:
    """

    response = llm.invoke(prompt)
    return response

# Final Test
test_query = "What did Dr. Aris Thorne find on Mars?"
answer = generate_answer(test_query)
print(answer)

StopIteration: 

# The End (of task 3)